In [ ]:
#Visualisations for model comparisons - for all steps of the nCV pipeline
%pip install -r "../Requirements.txt"

In [3]:
#Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import sys
import os

In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
results_baseline = Path("../results/baseline/all_families_summary.csv")
results_tuned = Path("../results/fs_tune_baseline/all_families_summary.csv")
results_final = Path("../results/final_model_per_family/all_families_summary.csv")


fig_dir = Path("../figures/visualisations")
fig_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
#Load data
df_baseline = pd.read_csv(results_baseline)
print(f"Baseline Results loaded successfully")
print(f"Families: {df_baseline['family'].unique().tolist()}")
print(f"Models: {df_baseline['model'].unique().tolist()}")

df_tuned = pd.read_csv(results_tuned)
print(f"Baseline Results loaded successfully")
print(f"Families: {df_tuned['family'].unique().tolist()}")
print(f"Models: {df_tuned['model'].unique().tolist()}")

df_final = pd.read_csv(results_final)
print(f"Baseline Results loaded successfully")
print(f"Families: {df_final['family'].unique().tolist()}")
print(f"Models: {df_final['model'].unique().tolist()}")

In [ ]:
families = ['KenyonCells', 'OpticLobe', 'Monoaminergic', 'Glia', 'Peptidergic', 'Clock']
ncols = 2
nrows = int(np.ceil(len(families) / ncols))


model_order = ["Dummy", "LinearRegression", "ElasticNet", "SVR", "RandomForest", "RandomForest_authors", "XGBoost"]

model_colors = {
    "Dummy":"#9AA5B1",
    "LinearRegression":"#B07AA1",
    "ElasticNet":"#4E79A7",
    "SVR": "#F28E2B",
    "RandomForest":"#59A14F",
    "RandomForest_authors": "#8CD17D",
    "XGBoost":"#E15759",
}

In [ ]:
#Figure 1 - Per family RMSE comparison with CI + dummy reference line
n_fam = len(families)
ncols = 3
nrows = int(np.ceil(n_fam / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4.2*nrows))
axes = np.array(axes).flatten()
 
for i, fam in enumerate(families):
    ax = axes[i]
    sub = df_baseline[df_baseline["family"] == fam].set_index("model").reindex(model_order)
 
    #dummy reference line
    dummy_rmse = sub.loc["Dummy", "RMSE_median"] if "Dummy" in sub.index else None
 
    y = np.arange(len(sub))
    vals = sub["RMSE_median"].values
    lo   = vals - sub["RMSE_lo"].values
    hi   = sub["RMSE_hi"].values - vals
    colors = [model_colors.get(m, "#888") for m in sub.index]
 
    ax.barh(y, vals, xerr=[lo, hi], color=colors, edgecolor="none",
            error_kw=dict(ecolor="#333", lw=1, capsize=3))
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=8)
    ax.invert_yaxis()
    if dummy_rmse is not None:
        ax.axvline(dummy_rmse, color="#C0392B", ls="--", lw=1,
                   label=f"Dummy ({dummy_rmse:.1f}d)")
        ax.legend(fontsize=7, loc="lower right")
    ax.set_title(f"{fam} — RMSE", fontsize=11)
    ax.set_xlabel("RMSE (days)")
 
for j in range(len(families), len(axes)):
    axes[j].axis("off")
 
plt.suptitle("Model comparison per family — RMSE with 95% CI", fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig(fig_dir / "01_Baseline_RMSE_per_family.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved 01__Baseline_RMSE_per_family.png")

In [ ]:
#Figure 2 - Per family R2 comparison with CI + dummy reference line
n_fam = len(families)
ncols = 3
nrows = int(np.ceil(n_fam / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4.2*nrows))
axes = np.array(axes).flatten()
 
for i, fam in enumerate(families):
    ax = axes[i]
    sub = df_baseline[df_baseline["family"] == fam].set_index("model").reindex(model_order)
 
    #dummy reference line
    dummy_r2 = sub.loc["Dummy", "R2_median"] if "Dummy" in sub.index else None
 
    y = np.arange(len(sub))
    vals = sub["R2_median"].values
    lo   = np.clip(vals - sub["R2_lo"].values, 0, None)
    hi   = np.clip(sub["R2_hi"].values - vals, 0, None)
    colors = [model_colors.get(m, "#888") for m in sub.index]
 
    ax.barh(y, vals, xerr=[lo, hi], color=colors, edgecolor="none",
            error_kw=dict(ecolor="#333", lw=1, capsize=3))
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=8)
    ax.invert_yaxis()
    if dummy_r2 is not None:
        ax.axvline(dummy_r2, color="#C0392B", ls="--", lw=1,
               label=f"Dummy ({dummy_r2:.2f})")    # .2f and no "d"
        ax.legend(fontsize=7, loc="lower right")
    ax.set_title(f"{fam} — R^2", fontsize=11)
    ax.set_xlabel("R^2")
 
for j in range(len(families), len(axes)):
    axes[j].axis("off")
 
plt.suptitle("Model comparison per family — R^2 with 95% CI", fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig(fig_dir / "02_Baseline_R2_per_family.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved 02__Baseline_R2_per_family.png")

In [ ]:
#Figure 3 - Per family Spearman comparison with CI + dummy reference line
n_fam = len(families)
ncols = 3
nrows = int(np.ceil(n_fam / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4.2*nrows))
axes = np.array(axes).flatten()

for i, fam in enumerate(families):
    ax = axes[i]
    sub = df_baseline[df_baseline["family"] == fam].set_index("model").reindex(model_order)

    #dummy reference line
    dummy_spear = sub.loc["Dummy", "Spearman_median"] if "Dummy" in sub.index else None

    y = np.arange(len(sub))
    vals = sub["Spearman_median"].values
    lo   = np.clip(vals - sub["Spearman_lo"].values, 0, None)
    hi   = np.clip(sub["Spearman_hi"].values - vals, 0, None)
    colors = [model_colors.get(m, "#888") for m in sub.index]

    ax.barh(y, vals, xerr=[lo, hi], color=colors, edgecolor="none",
            error_kw=dict(ecolor="#333", lw=1, capsize=3))
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=8)
    ax.invert_yaxis()
    if dummy_spear is not None:
        ax.axvline(dummy_spear, color="#C0392B", ls="--", lw=1,
                   label=f"Dummy ({dummy_spear:.2f})")
    ax.set_title(f"{fam} — Spearman", fontsize=11)
    ax.set_xlabel("Spearman correlation")
    ax.set_xlim(min(0, np.nanmin(vals) - 0.1), 1.0)

for j in range(len(families), len(axes)):
    axes[j].axis("off")

plt.suptitle("Model comparison per family — Spearman with 95% CI", fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig(fig_dir / "03_Baseline_Spearman_per_family.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved 03_Baseline_Spearman_per_family.png")

In [ ]:
#For the tuned models - comparison with the baseline models
def best_per_family(df, metric, lower_better=True):
    rows = []
    for fam in df["family"].unique():
        sub = df[(df["family"] == fam) & (df["model"] != "Dummy")]
        idx = sub[f"{metric}_median"].idxmin() if lower_better else sub[f"{metric}_median"].idxmax()
        best = sub.loc[idx]
        rows.append({"family": fam, "model": best["model"],
                     "value": best[f"{metric}_median"]})
    return pd.DataFrame(rows).set_index("family")
 
for metric, lower_better, label in [
    ("RMSE", True,  "RMSE (days)"),
    ("R2",   False, "R²"),
    ("Spearman", False, "Spearman correlation"),
]:
    base_best = best_per_family(df_baseline, metric, lower_better)
    tune_best = best_per_family(df_tuned,    metric, lower_better)
 
    fam_order = families
    x = np.arange(len(fam_order))
    base_vals = base_best.loc[fam_order, "value"].values
    tune_vals = tune_best.loc[fam_order, "value"].values
 
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - 0.2, base_vals, width=0.4, label="Baseline (best model)", color="#9AA5B1")
    ax.bar(x + 0.2, tune_vals, width=0.4, label="After FS + tuning",     color="#4E79A7")
 
    #annotate the improvement
    for i in range(len(fam_order)):
        if lower_better:
            improvement = base_vals[i] - tune_vals[i]
            txt = f"{improvement:+.1f}"
        else:
            improvement = tune_vals[i] - base_vals[i]
            txt = f"{improvement:+.2f}"
        ymax = max(base_vals[i], tune_vals[i])
        ax.text(x[i], ymax * 1.02, txt, ha="center", fontsize=8,
                color="#1F8A4C" if improvement > 0 else "#C0392B")
 
    ax.set_xticks(x); ax.set_xticklabels(fam_order, rotation=30, ha="right")
    ax.set_ylabel(label)
    ax.set_title(f"Baseline vs tuned — {metric}\n")
    ax.legend()
    plt.tight_layout()
    plt.savefig(fig_dir / f"A_baseline_vs_tuned_{metric}.png", dpi=300)
    plt.close()
    print(f"Saved A_baseline_vs_tuned_{metric}.png")

In [ ]:
METRICS = [
    ("RMSE", "RMSE (days)", True),
    ("R2", "R²",  False),
    ("Spearman", "Spearman correlation", False),
]

BASE_COLOR = "#9AA5B1"   
TUNE_COLOR = "#4E79A7"  

for metric, ylabel, lower_better in METRICS:
    fig, axes = plt.subplots(nrows, ncols, figsize=(8*ncols, 4.5*nrows))
    axes = np.array(axes).flatten()
 
    for i, fam in enumerate(families):
        ax = axes[i]
 
        base = df_baseline[df_baseline["family"] == fam].set_index("model")
        tune = df_tuned[df_tuned["family"] == fam].set_index("model")
 
        #models present in both
        models = [m for m in model_order if m in base.index and m in tune.index]
 
        x = np.arange(len(models))
        base_vals = base.loc[models, f"{metric}_median"].values
        tune_vals = tune.loc[models, f"{metric}_median"].values
 
        # error bars from CI columns
        base_lo = np.clip(base_vals - base.loc[models, f"{metric}_lo"].values, 0, None)
        base_hi = np.clip(base.loc[models, f"{metric}_hi"].values - base_vals, 0, None)
        tune_lo = np.clip(tune_vals - tune.loc[models, f"{metric}_lo"].values, 0, None)
        tune_hi = np.clip(tune.loc[models, f"{metric}_hi"].values - tune_vals, 0, None)
 
        ax.bar(x - 0.2, base_vals, width=0.4, yerr=[base_lo, base_hi],
               label="Baseline", color=BASE_COLOR, capsize=2,
               error_kw=dict(lw=0.8))
        ax.bar(x + 0.2, tune_vals, width=0.4, yerr=[tune_lo, tune_hi],
               label="Tuned (FS + HPO)", color=TUNE_COLOR, capsize=2,
               error_kw=dict(lw=0.8))
 
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=40, ha="right", fontsize=8)
        ax.set_title(fam, fontsize=12)
        ax.set_ylabel(ylabel)
        if not lower_better:
            #R2 / Spearman: cap at 1, allow slightly negative if present
            ymin = min(0, np.nanmin([base_vals.min(), tune_vals.min()]) - 0.1)
            ax.set_ylim(ymin, 1.0)
        ax.legend(fontsize=8)
 
    # hide unused panels
    for j in range(len(families), len(axes)):
        axes[j].axis("off")
 
    
    plt.suptitle(f"Baseline vs Tuned per family — {metric}",
                 fontsize=15, y=1.003)
    plt.tight_layout()
    out = fig_dir / f"baseline_vs_tuned_{metric}.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved {out.name}")
 
print(f"\nAll figures saved to: {fig_dir}")
 

In [ ]:
#Keep only XGBoost rows if the file has multiple models
if "model" in df_final.columns and "XGBoost" in df_final["model"].unique():
    xgb = df_final[df_final["model"] == "XGBoost"].set_index("family")
else:
    xgb = df_final.set_index("family")

METRICS = [
    ("RMSE", "RMSE (days)", "#E15759", True),
    ("R2", "R²", "#4E79A7", False),
    ("Spearman", "Spearman correlation", "#59A14F", False),
]

for metric, ylabel, color, ascending in METRICS:
    fam_order = xgb[f"{metric}_median"].sort_values(ascending=ascending).index.tolist()
    vals = xgb.loc[fam_order, f"{metric}_median"].values
    lo   = np.clip(vals - xgb.loc[fam_order, f"{metric}_lo"].values, 0, None)
    hi   = np.clip(xgb.loc[fam_order, f"{metric}_hi"].values - vals, 0, None)
 
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(fam_order))
    ax.bar(x, vals, yerr=[lo, hi], color=color, capsize=4, edgecolor="none")
    for i, v in enumerate(vals):
        offset = (max(vals) * 0.01)
        ax.text(i, v + hi[i] + offset, f"{v:.2f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(fam_order, rotation=30, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(f"XGBoost performance across families — {metric}")
    if metric in ("R2", "Spearman"):
        ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(fig_dir / f"B_xgboost_{metric}_across_families.png", dpi=300)
    plt.close()
    print(f"Saved B_xgboost_{metric}_across_families.png")

In [ ]:
#Combined figure
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (metric, ylabel, color, ascending) in zip(axes, METRICS):
    fam_order = xgb[f"{metric}_median"].sort_values(ascending=ascending).index.tolist()
    vals = xgb.loc[fam_order, f"{metric}_median"].values
    lo   = np.clip(vals - xgb.loc[fam_order, f"{metric}_lo"].values, 0, None)
    hi   = np.clip(xgb.loc[fam_order, f"{metric}_hi"].values - vals, 0, None)
    x = np.arange(len(fam_order))
    ax.bar(x, vals, yerr=[lo, hi], color=color, capsize=4, edgecolor="none")
    for i, v in enumerate(vals):
        ax.text(i, v + hi[i] + max(vals)*0.01, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(fam_order, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(metric)
    if metric in ("R2", "Spearman"):
        ax.set_ylim(0, 1)
plt.suptitle("XGBoost — performance across families", fontsize=15)
plt.tight_layout()
plt.savefig(fig_dir / "B_xgboost_all_metrics.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved B_xgboost_all_metrics.png")


## Generalization

In [ ]:
import warnings
import sys

sys.path.insert(0, str(Path.cwd().parent)) # import src/
warnings.filterwarnings("ignore")

project_dir = Path.cwd().parent

from src import build_xy, load_anndata, load_model

HVG_DATA_DIR = project_dir / 'data' / 'HVGs'
GENERALIZE_DATA_DIR = project_dir / 'data' / 'generalisability'
FIG_DIR = project_dir / 'visualisations' / 'figures'

FAMILIES = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells_train_hvg.h5ad",  HVG_DATA_DIR / "Kenyon_cells_test_hvg.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe_train_hvg.h5ad",    HVG_DATA_DIR / "Optic_lobe_test_hvg.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic_train_hvg.h5ad",HVG_DATA_DIR / "Monoaminergic_test_hvg.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia_train_hvg.h5ad",         HVG_DATA_DIR / "Glia_test_hvg.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic_train_hvg.h5ad",  HVG_DATA_DIR / "Peptidergic_test_hvg.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock_train_hvg.h5ad",        HVG_DATA_DIR / "Clock_test_hvg.h5ad"),
}
FAMILIES = {f: p for f, p in FAMILIES.items() if p[0].exists() and p[1].exists()}

FAMILIES_FEMALE = {
    "KenyonCells":  (GENERALIZE_DATA_DIR / "Kenyon_cells__sex_train_female_test_male_train.h5ad",  GENERALIZE_DATA_DIR / "Kenyon_cells__sex_train_female_test_male_test.h5ad"),
    "OpticLobe":    (GENERALIZE_DATA_DIR / "Optic_lobe__sex_train_female_test_male_train.h5ad",    GENERALIZE_DATA_DIR / "Optic_lobe__sex_train_female_test_male_test.h5ad"),
    "Monoaminergic":(GENERALIZE_DATA_DIR / "Monoaminergic__sex_train_female_test_male_train.h5ad", GENERALIZE_DATA_DIR / "Monoaminergic__sex_train_female_test_male_test.h5ad"),
    "Glia":         (GENERALIZE_DATA_DIR / "Glia__sex_train_female_test_male_train.h5ad",          GENERALIZE_DATA_DIR / "Glia__sex_train_female_test_male_test.h5ad"),
    "Peptidergic":  (GENERALIZE_DATA_DIR / "Peptidergic__sex_train_female_test_male_train.h5ad",   GENERALIZE_DATA_DIR / "Peptidergic__sex_train_female_test_male_test.h5ad"),
    "Clock":        (GENERALIZE_DATA_DIR / "Clock__sex_train_female_test_male_train.h5ad",         GENERALIZE_DATA_DIR / "Clock__sex_train_female_test_male_test.h5ad"),
}
FAMILIES_FEMALE = {f: p for f, p in FAMILIES_FEMALE.items() if p[0].exists() and p[1].exists()}

FAMILIES_MALE = {
    "KenyonCells":  (GENERALIZE_DATA_DIR / "Kenyon_cells__sex_train_male_test_female_train.h5ad",  GENERALIZE_DATA_DIR / "Kenyon_cells__sex_train_male_test_female_test.h5ad"),
    "OpticLobe":    (GENERALIZE_DATA_DIR / "Optic_lobe__sex_train_male_test_female_train.h5ad",    GENERALIZE_DATA_DIR / "Optic_lobe__sex_train_male_test_female_test.h5ad"),
    "Monoaminergic":(GENERALIZE_DATA_DIR / "Monoaminergic__sex_train_male_test_female_train.h5ad", GENERALIZE_DATA_DIR / "Monoaminergic__sex_train_male_test_female_test.h5ad"),
    "Glia":         (GENERALIZE_DATA_DIR / "Glia__sex_train_male_test_female_train.h5ad",          GENERALIZE_DATA_DIR / "Glia__sex_train_male_test_female_test.h5ad"),
    "Peptidergic":  (GENERALIZE_DATA_DIR / "Peptidergic__sex_train_male_test_female_train.h5ad",   GENERALIZE_DATA_DIR / "Peptidergic__sex_train_male_test_female_test.h5ad"),
    "Clock":        (GENERALIZE_DATA_DIR / "Clock__sex_train_male_test_female_train.h5ad",         GENERALIZE_DATA_DIR / "Clock__sex_train_male_test_female_test.h5ad"),
}
FAMILIES_MALE = {f: p for f, p in FAMILIES_MALE.items() if p[0].exists() and p[1].exists()}



In [10]:
def plot_eval_metrics(res, metric, path):
    color = {"RMSE": "#d65f5f", "MAE": "#7e57c2"}.get(metric, "#4c72b0")

    series = {title: pd.Series({fam: d[metric] for fam, d in scores.items()})
              for title, scores in res.items()}

    first = next(iter(series.values()))
    fam_order = first.sort_values(ascending=True).index.tolist()

    n = len(series)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), sharey=True)
    if n == 1:
        axes = [axes]

    ymax = max(s.max() for s in series.values()) * 1.15   # shared headroom for labels

    for ax, (title, s) in zip(axes, series.items()):
        vals = s.reindex(fam_order).values
        x = np.arange(len(fam_order))
        ax.bar(x, vals, color=color, edgecolor="none")
        for i, v in enumerate(vals):
            if not np.isnan(v):
                ax.text(i, v + ymax * 0.01, f"{v:.2f}", ha="center", fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(fam_order, rotation=40, ha="right", fontsize=8)
        ax.set_title(title); ax.set_ylim(0, ymax)
    axes[0].set_ylabel(f"{metric} (days)")

    plt.suptitle(f"XGBoost — {metric} across evaluation scenarios", fontsize=15)
    plt.tight_layout(); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()
    print(f"Saved {metric} to {path}")

In [25]:
# Predict on male test sets
test_scores_female_male = {}
for fam, (train, test) in FAMILIES_FEMALE.items():
    Xte, yte, _ = build_xy(load_anndata(test), extra_obs=("nUMI",))
    pipe_female_eval = load_model(f"../results/final_model_per_family/{fam}/final_XGBoost_generalize_sex_female.joblib")
    pred_male = pipe_female_eval.predict(Xte)
    test_scores_female_male[fam] = {"MAE": float(np.mean(np.abs(yte - pred_male))),
                                    "RMSE": float(np.sqrt(np.mean((yte - pred_male) ** 2)))}
# display(test_scores_female_male)

# Predict on female test sets
test_scores_male_female = {}
for fam, (train, test) in FAMILIES_MALE.items():
    Xte, yte, _ = build_xy(load_anndata(test), extra_obs=("nUMI",))
    pipe_male_eval = load_model(f"../results/final_model_per_family/{fam}/final_XGBoost_generalize_sex_male.joblib")
    pred_female = pipe_male_eval.predict(Xte)
    test_scores_male_female[fam] = {"MAE": float(np.mean(np.abs(yte - pred_female))),
                                    "RMSE": float(np.sqrt(np.mean((yte - pred_female) ** 2)))}
# display(test_scores_male_female)

# Predict on test sets
test_scores = {}
for fam, (train, test) in FAMILIES.items():
    Xte, yte, _ = build_xy(load_anndata(test), extra_obs=("nUMI",))
    pipe = load_model(f"../results/final_model_per_family/{fam}/final_XGBoost.joblib")
    pred = pipe.predict(Xte)
    test_scores[fam] = {"MAE": float(np.mean(np.abs(yte - pred))),
                        "RMSE": float(np.sqrt(np.mean((yte - pred) ** 2)))}
# display(test_scores)

In [28]:
res = {
    "All data train and test": test_scores,
    "Train on female test on male": test_scores_female_male,
    "Train on male test on female": test_scores_male_female,
}

In [29]:
plot_eval_metrics(res=res, metric="RMSE", path=FIG_DIR / "eval_RMSE.png")
plot_eval_metrics(res=res, metric="MAE", path=FIG_DIR / "eval_MAE.png")

Saved RMSE to /home/giorgos/Documents/MLCB/MLCB_project/figures/eval_RMSE.png
Saved MAE to /home/giorgos/Documents/MLCB/MLCB_project/figures/eval_MAE.png
